<a href="https://colab.research.google.com/github/hiiamlars/master_thesis/blob/main/llm_reviews.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Setup

In [67]:
# @title Dependencies

# Installation
!pip install pandas pyarrow -q
!pip install numpy -q
!pip install tqdm -q
!pip install openai -q
!pip install groq -q

# First-party installations
import itertools
import re
import json
import random
import time
from typing import Dict, Any, Tuple, Optional, Literal, List
from dataclasses import dataclass
import os

# Third-party installations
from google.colab import drive
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
from openai import OpenAI
from openai import APIError
from groq import Groq

In [ ]:
# @title Paths and definitions

# Mount GDrive
drive.mount('/content/drive')

# Set working directory
WORKING_DIR = "/content/drive/MyDrive/Thesis/"

# Create working directory (if not already existing)
os.makedirs(WORKING_DIR, exist_ok=True)
print(f"Ensured working directory exists at: {WORKING_DIR}.")

# Define and create DATASET_DIR to load the paper content
DATASET_DIR = os.path.join(WORKING_DIR, "iclr/final")
os.makedirs(DATASET_DIR, exist_ok=True)
print(f"Ensured dataset directory exists at: {DATASET_DIR}.")

# Define and create the RAW_DIR for the llm reviews (and their logs)
RAW_DIR = os.path.join(WORKING_DIR, "llm_reviews")
os.makedirs(RAW_DIR, exist_ok=True)
print(f"Ensured raw directory exists at: {RAW_DIR}.")

In [69]:
# @title Hardcoded definitions

# Models using the OpenAI-API (as used by Cummins 2025)
OPENAI_MODELS = [
    "gpt-5-mini-2025-08-07",
    "gpt-5-nano-2025-08-07",
    "gpt-4o-2024-08-06",
    "gpt-4o-mini-2024-07-18",
    "o4-mini-2025-04-16",
    "o3-mini-2025-01-31",
    "gpt-3.5-turbo-0125"
]

# Models using the Groq-API (as used by Cummins 2025)
GROQ_MODELS = [
    "llama-3.3-70b-versatile",
    "deepseek-r1-distill-llama-70b"
]

# Models who can vary by reasoning effort
REASONING_MODELS = {
    "o4-mini-2025-04-16",
    "gpt-5-mini-2025-08-07",
    "o3-mini-2025-01-31",
    "gpt-5-nano-2025-08-07"
}

# Define temperature parameters (as used by Cummins 2025)
TEMPS = [0.0, 0.5, 1.0, 1.5]

# Define reasoning parameters (as used by Cummins 2025)
REASONING = ["low", "high"]

# Define prompt style (CoT) (as used by Li et al. 2025)
CoT = [
    "",
    "Explain your thought process step by step."
]

# Define scope
SCOPE = [
  "abstract",
  "full_text"
]

# Define the number of reviews to generate per paper for each combo
NUM_REVIEWS_PER_PAPER = 2

# Create data class for parameter combinations
@dataclass
class ParamCombo:
    model: str
    temperature: Optional[float]
    reasoning_effort: Optional[Literal[*REASONING]]
    chain_of_thought: Literal[*CoT]
    scope: Literal[*SCOPE]

# Define the full result path
RESULTS_PATH = os.path.join(RAW_DIR, "llm_sim_results.parquet")

# Define the full log path
LOG_PATH = os.path.join(RAW_DIR, "llm_sim_progress.parquet")

INCLUDE_NOTE_TAKING = True # Set to False to deactivate the note-taking step
SIMULATION_ACTIVE = True # SIMULATION_ACTIVE = False activates the API calls

In [70]:
# @title APIs

NOTES_PROMPT = """

Here is an ICLR paper: {{text}}

Take notes on the paper for an ICLR style review.

""" # Adopted and adjusted from Robertson and Koyejo (2025)

NOTE_TAKING_CONFIG = { "task_type": "peer_review",
                      "task_description": "Scientific paper peer review task",
                       "model_config":
                        { "model_name": "gpt-4o-mini",
                         "max_tokens": 2000,
                          "temperature": 0.7 }
} # Adopted and adjusted from Robertson and Koyejo (2025)

LLM_PROMPT = """

Here is the text of a paper:

{{text}}

Create an ICLR-style review following this specific structure:

# Summary Of The Paper

Summarize the paper’s main contributions, methodology, and findings.

# Strength And Weaknesses

Analyze the paper’s contributions based on your notes.

# Clarity, Quality, Novelty And Reproducibility

Evaluate based on your notes.

# Summary Of The Review

Provide a 2-3 sentence distillation of your overall assessment.

# Correctness

Rate on a scale of 1-5.

# Technical Novelty And Significance

Rate on a scale of 1-5.

# Empirical Novelty And Significance

Rate on a scale of 1-5.

Maintain a professional tone throughout. Base your review entirely on your reading notes.

""" # Adopted and adjusted from Robertson and Koyejo (2025)

# OpenAI API-key
OPENAI_KEY = "api_key_placeholder"

# Groq API-key
GROQ_KEY = "api_key_placeholder"

# Groq URL
GROQ_URL ="https://api.groq.com/openai/v1"

# Settings
MAX_RETRIES = 3 # Random
RETRY_DELAY = 1.5 # Random

In [71]:
# @title Load data

# Read dataset_paper
dataset_paper = pd.read_parquet(os.path.join(DATASET_DIR, "dataset_paper.parquet"))

# Select target columns
paper_content = dataset_paper[['paper_id', 'abstract', 'parsed_text']]

# Check paper_content
print(paper_content.head(3))

# Prepare Dict for later applications
targets = paper_content.to_dict('records')

      paper_id                                           abstract  \
0  vuD2xEtxZcj  In deep learning, fine-grained N:M sparsity re...   
1  WoByU5W5te0  We present a novel method to regularizes neura...   
2  XZRmNjUMj0c  Neural Architecture Search (NAS) is a fast-dev...   

                                         parsed_text  
0  MINIMUM VARIANCE UNBIASED N:M SPARSITY FOR THE...  
1  GECONERF: FEW-SHOT NEURAL RADIANCE FIELDS VIA ...  
2  EFFICIENT ONE-SHOT NEURAL ARCHITECTURE SEARCH ...  


### Response handling

In [72]:
def _schema_to_tool(chain_of_thought: str) -> Dict[str, Any]:
    """
    Creates a response schema for API calls with certain models.
    Additionally it adds the chain of thought instruction to the prompt.
    """

    return {
        "type": "function",
        "function": {
            "name": "response_format",
            "description": "The function used to return the single, structured text response.",
            "parameters": {
                "type": "object",
                "properties": {
                    "final_response": {
                        "type": "string",
                        "description": (
                            f"Generate the complete, continuous peer-review based on the provided content. {chain_of_thought}"
                        )
                    }
                },
                "required": ["final_response"]
            }
        }
    }

def _extract_json(text: str, model_name: str = "") -> dict:
    """Extract and parse a JSON object from a potentially messy API response"""
    if not text:
        return {}

    # DeepSeek R1: strip 'thinking' section if present
    if "deepseek-r1-distill-llama-70b" in model_name:
        # Common pattern: <think>...</think>{...} or reasoning text before JSON
        if "</think>" in text:
            text = text.split("</think>", 1)[-1]
        elif "<think>" in text:
            text = text.split("<think>", 1)[-1]
        # Otherwise, try to cut at the first { or [ after any long text
        if "{" in text or "[" in text:
            idx = min(
                [i for i in [text.find("{"), text.find("[")] if i != -1]
            )
            text = text[idx:]

    # Strip code fences if present
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text)
        text = re.sub(r"\s*```$", "", text)

    # Find first {...}
    m = re.search(r"(\{.*\})", text, flags=re.DOTALL)
    if m:
        try:
            return json.loads(m.group(1))
        except Exception:
            pass

    try:
        return json.loads(text)
    except Exception:
        return {}

### API

In [73]:
class LLMClient:
    def __init__(self, simulate: bool = True, seed: int = 7, include_note_taking: bool = True):
        """Initialize LLMClient with defaults"""
        self.simulate = simulate
        self.rng = random.Random(seed)
        self.include_note_taking = include_note_taking

        # Placeholders for real clients when simulate=False:
        self._openai = None
        self._groq = None

    def _maybe_init_clients(self):
        """Start simulation or API clients"""
        if self.simulate:
            return
        self._openai = OpenAI(api_key=OPENAI_KEY)
        self._groq = Groq(api_key=GROQ_KEY)

    def call(
        self,
        model: str,
        paper_content: str,
        temperature: float,
        reasoning_effort: str,
        chain_of_thought: str,
        max_retries: float = MAX_RETRIES,
        retry_delay: float = RETRY_DELAY,
        ) -> Tuple[str, Dict]:

        """
        Define the API calls / simulations according to the input parameters or hardcoded configurations.
        If note taking is active, execute this step as well, before the llm reviews are generated.
        If the note taking fails do not attempt to generate llm reviews.
        """

        self._maybe_init_clients()

        # Default to using paper_content directly if note-taking is skipped
        notes_output = paper_content

        if self.include_note_taking:
            # --- Step 1: Generate notes using NOTES_PROMPT ---

            # Extract configurations
            notes_model_name = NOTE_TAKING_CONFIG["model_config"]["model_name"]
            notes_temperature = NOTE_TAKING_CONFIG["model_config"]["temperature"]
            notes_max_tokens = NOTE_TAKING_CONFIG["model_config"]["max_tokens"]

            # Generate prompt
            notes_prompt_text = NOTES_PROMPT.replace("{{text}}", paper_content)

            # In simulation create artificial output
            if self.simulate:
                notes_output = (
                    f"Simulated Notes: Model='{notes_model_name}', Temp={notes_temperature}, "
                    f"Max Tokens='{notes_max_tokens}', Paper excerpt='{paper_content[:100]}..."
                )
            # In active mode try successful calls till max_retries
            else:
                for attempt_notes in range(1, max_retries + 1):
                    try:
                        msgs_notes = [{"role": "user", "content": notes_prompt_text}]
                        resp_notes = self._openai.chat.completions.create(
                            model=notes_model_name,
                            messages=msgs_notes,
                            temperature=notes_temperature,
                            max_tokens=notes_max_tokens,
                        )
                        notes_output = resp_notes.choices[0].message.content.strip()
                        break # If successful, break retry loop
                    except Exception as e:
                        print(f"[LLM ERROR] model={notes_model_name} attempt={attempt_notes} -> {e}", flush=True)
                        if attempt_notes == max_retries:
                            notes_output = f"ERROR_NOTES: {type(e).__name__}: {e}" # Error as output for total failure
                        time.sleep(retry_delay)

            # If note-taking failed, skip the review generation completly
            if notes_output.startswith("ERROR_NOTES:"):
                return notes_output, {}

        # --- Step 2: Generate main review using LLM_PROMPT with the generated notes or original content ---
        final_review_prompt = LLM_PROMPT.replace("{{text}}", notes_output)

        # Append chain_of_thought instruction to the final prompt for the main review
        if chain_of_thought:
            final_review_prompt += f"\n\n{chain_of_thought}"

        provider = "groq" if model in GROQ_MODELS else "openai"

        for attempt in range(1, max_retries + 1):
            try:
                if self.simulate:
                    # Construct artificial output for the main review
                    raw = {
                        "model": model,
                        "temperature": temperature,
                        "reasoning_effort": reasoning_effort,
                        "chain_of_thought": chain_of_thought,
                        "simulated_review": (
                            f"This is a simulated review: Model='{model}', Temp={temperature}, "
                            f"Effort='{reasoning_effort}', chain_of_thought='{chain_of_thought}'. "
                            f"Based on notes: '{notes_output[:50]}..."
                        )
                    }
                    structured = json.dumps(raw, indent=2)
                    return structured, raw

                # For OpenAI models
                if provider == "openai":
                    # For their reasoning models
                    if model in REASONING_MODELS:
                        kwargs = {}
                        if reasoning_effort:
                            kwargs["reasoning"] = {"effort": reasoning_effort}
                        if temperature is not None:
                            kwargs["temperature"] = float(temperature)

                        contract = "Return exactly ONE JSON object"
                        resp = self._openai.responses.create(
                            model=model,
                            input=(final_review_prompt + contract),
                            **kwargs,
                        )
                        raw = getattr(resp, "output_text", None) or str(resp)

                    # For their non-reasoning models
                    else:
                        tools = [_schema_to_tool(chain_of_thought)]
                        ckwargs = {}
                        if temperature is not None:
                            ckwargs["temperature"] = float(temperature)

                        system_msg = {
                            "role": "system",
                            "content": "You are a parser. Call the function with exactly one JSON object that includes a text."
                        }
                        msgs = [system_msg, {"role": "user", "content": final_review_prompt}]
                        resp = self._openai.chat.completions.create(
                            model=model,
                            messages=msgs,
                            tools=tools if tools else None,
                            **ckwargs,
                        )
                        choice = resp.choices[0]
                        if choice.message.tool_calls:
                            args = choice.message.tool_calls[0].function.arguments
                            raw = args
                        else:
                            raw = choice.message.content or ""

                # For GROQ (non OpenAI) models
                else:
                    contract = "Return exactly ONE JSON object"
                    prompt2 = final_review_prompt + contract
                    gkwargs = {}
                    if temperature is not None and model not in REASONING_MODELS:
                        gkwargs["temperature"] = float(temperature)
                    resp = self._groq.chat.completions.create(
                        model=model,
                        messages=[{"role": "user", "content": prompt2}],
                        **gkwargs,
                    )
                    raw = resp.choices[0].message.content.strip()

                parsed_raw = _extract_json(raw, model_name=model)
                return raw, parsed_raw

            # Error handling
            except Exception as e:
                print(f"[LLM ERROR] provider={provider} model={model} attempt={attempt} -> {e}", flush=True)
                if attempt == max_retries:
                    err_stub = f"ERROR: {type(e).__name__}: {e}" # Error as output for total failure
                    return err_stub, {}

                # Delay until next attempt (if max_retries has not been reached yet)
                time.sleep(retry_delay)

### Helper for output messages

In [74]:
def _fmt_combo(combo: ParamCombo) -> str:
    """Helper to format the ParamCombo into a human-readable string"""
    parts = []
    if combo.model is not None:
        parts.append(f"model={combo.model}")
    if combo.temperature is not None:
        parts.append(f"temperature={combo.temperature}")
    if combo.reasoning_effort is not None:
        parts.append(f"reasoning_effort={combo.reasoning_effort}")
    # Chain_of_thought can be an empty string
    parts.append(f"chain_of_thought={combo.chain_of_thought}")
    if combo.scope is not None:
        parts.append(f"scope={combo.scope}")
    return ", ".join(parts)

def log_call(doc_id: str, combo: ParamCombo, **context: Dict[str, Any]):
    """
    Another helper to enrich the ParamCombo string with information for output messages"
    """
    ctx = ", ".join(f"{k}={v}" for k, v in context.items() if v is not None)
    msg = f"[LLM CALL] paper={doc_id} | {_fmt_combo(combo)}"
    if ctx:
        msg += f" | {ctx}"
    print(msg, flush=True)

### Simulation definition

In [75]:
def simulate_parameter_combo(
    combo: ParamCombo,
    review_targets: List[Dict],
    client: Optional[LLMClient] = None,
) -> pd.DataFrame:

    """
    Iterates through the papers, creates the final user prompt,
    takes the configurational setting,
    calls the API/simulation NUM_REVIEWS_PER_PAPER times
    and collects the resulting reviews in a pandas DataFrame
    """

    # If client specified then use client, otherwise fake client for test
    client = client or LLMClient(simulate=True)

    records = []
    for review_item in review_targets:
      # Select content based on scope
      if combo.scope == "full_text":
          paper_content_for_llm = review_item['parsed_text']
      elif combo.scope == "abstract":
          paper_content_for_llm = review_item['abstract']
      else:
          raise ValueError(f"Invalid scope: {combo.scope}. Must be 'full_text' or 'abstract'.")

      doc_id = review_item["paper_id"]

      # Initialize a dictionary for the current paper's results, including combo parameters
      current_paper_record = {
          "paper_id": doc_id,
          "model": combo.model,
          "temperature": combo.temperature,
          "reasoning_effort": combo.reasoning_effort,
          "chain_of_thought": combo.chain_of_thought,
          "scope": combo.scope
      }

      # Loop to generate multiple reviews for each paper and parameter combo
      for review_idx in range(NUM_REVIEWS_PER_PAPER):

          log_call(doc_id, combo, review_number=review_idx + 1)

          raw, parsed = client.call(
              model=combo.model,
              paper_content=paper_content_for_llm,
              temperature=(combo.temperature if combo.model not in REASONING_MODELS else None),
              reasoning_effort=(combo.reasoning_effort if combo.model in REASONING_MODELS else None),
              chain_of_thought=combo.chain_of_thought
          )

          # Store each raw response in a new column, e.g., 'llm_review_1', 'llm_review_2'
          current_paper_record[f"llm_review_{review_idx + 1}"] = raw
          # Store the parsed response, if available, otherwise None
          current_paper_record[f"parsed_llm_review_{review_idx + 1}"] = parsed.get('final_response')

      # Append the single record for the current paper to the list of records
      records.append(current_paper_record)

    df = pd.DataFrame.from_records(records)
    return df

### Configuration grid

In [ ]:
# Complete list of models
MODELS = OPENAI_MODELS + GROQ_MODELS

# Generates every single combination
raw_grid = list(itertools.product(MODELS, TEMPS, REASONING, CoT, SCOPE))
grid_df = pd.DataFrame(raw_grid, columns=["model", "temperature", "reasoning_effort", "chain_of_thought", "scope"])

# Conditional deletion whether a model has a temperature or reasoning parameter
grid_df["reasoning_effort"] = np.where(
    ~grid_df["model"].isin(REASONING_MODELS), np.nan, grid_df["reasoning_effort"]
)
grid_df["temperature"] = np.where(
    grid_df["model"].isin(REASONING_MODELS), np.nan, grid_df["temperature"]
)

# Drop duplicates
experiment_config = grid_df.drop_duplicates(ignore_index=True)

print(f"Total valid parameter combos: {len(experiment_config)}")

display(experiment_config)

### Helper for logging

In [77]:
def _nan_to_none(x):

    """Use pandas' isna to catch float('nan'), numpy.nan, pd.NA, and None"""

    try:
        if pd.isna(x):
            return None
    except Exception:
        pass
    return x


def combo_key_tuple(row) -> tuple:

    """Combines the congfigurational setting with None instead of NaN"""

    return (
        row["model"],
        _nan_to_none(row["temperature"]),
        _nan_to_none(row["reasoning_effort"]),
        row["chain_of_thought"],
        row["scope"]
    )

def combo_key_str(row) -> str:

    """Creates readable combo string for logging"""

    t = combo_key_tuple(row)
    # The string format needs to match the tuple elements
    return f"model={t[0]}|temp={t[1]}|re={t[2]}|cot={t[3]}|scope={t[4]}"

### LLM review generation

In [ ]:
if __name__ == '__main__':

    # Initialize keys
    done_keys = set()

    # Check and read successful combos
    if os.path.exists(LOG_PATH):
        try:
            log_df = pd.read_parquet(LOG_PATH)
            if 'completed_combo' in log_df.columns:
                done_keys.update(log_df['completed_combo'].tolist())
            else:
                print(f"Warning: '{LOG_PATH}' exists but does not contain 'completed_combo' column. Starting with empty log.")
        except Exception as e:
            print(f"Error reading existing log parquet file: {e}. Starting with empty log.")
            done_keys = set()

    if os.path.exists(RESULTS_PATH):
        existing_df = pd.read_parquet(RESULTS_PATH)
        if not existing_df.empty:
            expected_rows_per_combo = len(targets)

            # Normalize NA for comparison
            ex = existing_df.copy()
            # Ensure these columns exist (they should)
            needed_col = ["model","temperature","reasoning_effort","chain_of_thought","scope"]
            missing_cols = [c for c in needed_col if c not in ex.columns]
            if not missing_cols:
                # Normalize NAs to None for keys
                for c in ["temperature","reasoning_effort"]:
                    ex[c] = ex[c].where(pd.notna(ex[c]), None)

                # Check if all review columns exist
                review_cols_present = all(f'llm_review_{i}' in ex.columns for i in range(1, NUM_REVIEWS_PER_PAPER + 1))

                if review_cols_present:
                    # Define the success checking function
                    def is_successful_combo(group):
                        for i in range(1, NUM_REVIEWS_PER_PAPER + 1):
                            review_col = f'llm_review_{i}'
                            # Check if any review starts with an error indicator
                            if group[review_col].astype(str).str.startswith(('ERROR_NOTES:', 'ERROR:')).any():
                                return False
                        return True

                  # Apply helper functions per groups
                    counts = (
                        ex.groupby(needed_col)
                        .apply(lambda x: pd.Series({
                            'nrows': len(x),
                            'is_successful': is_successful_combo(x)
                        }), include_groups=False)
                        .reset_index()
                    )

                    # Only mark as done if all expected rows for that combo are present AND all reviews are successful
                    for _, r in counts.iterrows():
                        if int(r["nrows"]) == expected_rows_per_combo and r['is_successful']:
                            done_keys.add(combo_key_str(r))

    print(f"Already-completed combos: {len(done_keys)}")

    # Initialize client
    client = LLMClient(simulate=SIMULATION_ACTIVE, include_note_taking=INCLUDE_NOTE_TAKING)

    new_rows_count = 0
    for idx, row in tqdm(grid_df.head(5).iterrows(), total=len(grid_df.head(5)), desc="Processing Combos"): # for test reasons only the first x rows

        # Check if key is already completed
        key = combo_key_str(row)

        if key in done_keys:
            print(f"[SKIP] {key} already complete.")
            continue

        # If not, construct current combo
        combo = ParamCombo(
            model=row["model"],
            temperature=None if pd.isna(row["temperature"]) else float(row["temperature"]),
            reasoning_effort=None if pd.isna(row["reasoning_effort"]) else str(row["reasoning_effect"]),
            chain_of_thought=row["chain_of_thought"],
            scope=row["scope"]
        )

        print(f"\n[RUN] {idx+1}/{len(grid_df)} -> {key}", flush=True)
        try:

            # Make actual simulation / call
            df_combo = simulate_parameter_combo(combo, targets, client=client)

            if os.path.exists(RESULTS_PATH):
                # If result file exist, read, concat and save
                existing_df = pd.read_parquet(RESULTS_PATH)
                combined_df = pd.concat([existing_df, df_combo], ignore_index=True)
                combined_df.to_parquet(RESULTS_PATH, index=False)
            else:
                # Else create a new result file
                df_combo.to_parquet(RESULTS_PATH, index=False)

            # Convert key after successful result storage
            new_log_entry_df = pd.DataFrame([key], columns=['completed_combo'])

            if os.path.exists(LOG_PATH):
                try:
                    # If log file exists, read, concat, drop duplicates and save
                    existing_log_df = pd.read_parquet(LOG_PATH)
                    combined_log_df = pd.concat([existing_log_df, new_log_entry_df], ignore_index=True)
                    combined_log_df.drop_duplicates(subset=['completed_combo'], inplace=True)
                    combined_log_df.to_parquet(LOG_PATH, index=False)

                    # Continue with next iteration only after successful result AND log save
                    new_rows_count += len(df_combo)

                    # Add key only after successful result AND log save
                    done_keys.add(key)
                except Exception as e:
                    print(f"Warning: Could not append to existing log parquet file {LOG_PATH}: {e}. Skipping log entry for this combo.")
            else:
                # Else write a new log file
                new_log_entry_df.to_parquet(LOG_PATH, index=False)

                # Continue with next iteration only after successful result AND log save
                new_rows_count += len(df_combo)

                # Add key only after successful result AND log save
                done_keys.add(key)

        except Exception as e:
            # In case of failure do not save anything and print this error message
            print(f"[ERROR] {key} -> {type(e).__name__}: {e}", flush=True)

In [ ]:
# result.parquet-file
tvd_mi_results_df = pd.read_parquet(RESULTS_PATH)

# Check results
display(tvd_mi_results_df.head(5))

# log.parquet-file
if os.path.exists(LOG_PATH):
    try:
        tvd_mi_log_df = pd.read_parquet(LOG_PATH)
        # Check if the expected column exists, otherwise display full dataframe
        if 'completed_combo' in tvd_mi_log_df.columns:
            display(tvd_mi_log_df[['completed_combo']].head(5))
        else:
            print(f"Warning: Log file '{LOG_PATH}' exists but does not contain 'completed_combo' column.")
            display(tvd_mi_log_df.head(5))
    except Exception as e:
        print(f"Error reading log parquet file for display: {e}")
else:
    print(f"Log file '{LOG_PATH}' does not exist.")

### References

The main code logic and in parts exact code has been taken from:

Cummins, J. (2025). The threat of analytic flexibility in using large language models to simulate human data: A call to attention. *arXiv preprint* arXiv:2509.13397.